# Dataset Raw de la Premier League - EDA

Análisis Exploratorio de Datos (EDA) de las temporadas de la Premier League (`2014-15` a `2023-24`) a partir de datasets de partidos en formato raw procedentes de `football-data.co.uk`.

## Objetivos
- Comprender la estructura del dataset y la estabilidad del esquema entre temporadas.
- Evaluar la calidad e integridad de los datos (identificador de liga, valores nulos, duplicados y rangos inválidos).
- Identificar posibles riesgos antes de avanzar a la fase de limpieza de datos.

## Estructura del Notebook

El análisis se organiza en las siguientes secciones:

**0. Entorno y configuración**  
Carga de librerías y definición de rutas del proyecto.

**1. Preparación y organización de datos**  
Identificación de ficheros disponibles y construcción del diccionario de temporadas.

**2. Consistencia del esquema de datos**  
Comparación de dimensiones, definición del core dataset y verificación de estabilidad en tipos de dato.

**3. Calidad de datos**  
Evaluación de la completitud del core dataset por variable y temporada.

**4. Validaciones de integridad**  
Comprobaciones de coherencia en el core dataset (identificador de liga, resultados, nombres de equipos, duplicados y valores no válidos).

**5. Variables de cuotas**  
Identificación de columnas de apuestas, análisis de cobertura y validación de rangos.

**6. Conclusiones del análisis exploratorio**  
Síntesis estructural y consideraciones para la fase de limpieza.

**7. Exportación del core dataset**  
Montaje y guardado del core dataset validado para su uso en el análisis comparativo y en las fases posteriores del pipeline.

## 0) Entorno y configuración

En esta sección se configuran las dependencias, librerías y parámetros globales necesarios para garantizar la reproducibilidad del análisis.

In [54]:
from pathlib import Path
import sys
import json
import pandas as pd
from IPython.display import display

# Configuración de rutas del proyecto
PROJECT_ROOT = Path.cwd().parents[1]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_ROOT = PROJECT_ROOT / "config"
with open(CONFIG_ROOT / "leagues.json") as f:
    ALL_LEAGUES = json.load(f)

# Importación de funciones propias
from src.analysis import check_name_consistency, group_columns

### Configuración específica de la liga

In [55]:
LEAGUE = "premier" 
FILE_PREFIX = LEAGUE

RAW_DIR = PROJECT_ROOT / "data" / "raw" / LEAGUE
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / LEAGUE
GLOB_PATTERN = f"{FILE_PREFIX}_*_raw.csv"

CORE_DIR = PROCESSED_DIR / "core_validated.parquet"
METADATA_DIR = PROCESSED_DIR / "core_schema.json"

## 1) Preparación y organización de datos

En esta fase se identifican y organizan los datos por temporada con el fin de establecer una estructura coherente previa al análisis estructural.


### 1.1 Carga de datos y verificación inicial

Se identifican los archivos CSV disponibles y se comprueba la disponibilidad de temporadas suficientes para el análisis.

In [56]:
RAW_DIR = PROJECT_ROOT / "data" / "raw" / LEAGUE
csv_files = sorted(RAW_DIR.glob(GLOB_PATTERN))

if len(csv_files) < 2:
    raise ValueError(f'Se esperaban al menos 2 ficheros en {RAW_DIR}, encontrados {len(csv_files)}')

print(f"Directorio analizado: {RAW_DIR}")
print(f"Ficheros detectados: {len(csv_files)}")

Directorio analizado: /Users/jorgepais/Desktop/kraken/football-analytics/data/raw/premier
Ficheros detectados: 10


### 1.2 Inspección de ficheros disponibles

Se listan los ficheros detectados para verificar las temporadas disponibles antes de su carga.

In [57]:
names = [p.name for p in csv_files]

HEAD_N = 5
print("Muestra de ficheros:")

if len(names) <= 2 * HEAD_N:
    for n in names:
        print(f"  - {n}")
else:
    for n in names[:HEAD_N]:
        print(f"  - {n}")
    print(f"  ... ({len(names) - 2 * HEAD_N} ficheros omitidos) ...")
    for n in names[-HEAD_N:]:
        print(f"  - {n}")

Muestra de ficheros:
  - premier_2014_15_raw.csv
  - premier_2015_16_raw.csv
  - premier_2016_17_raw.csv
  - premier_2017_18_raw.csv
  - premier_2018_19_raw.csv
  - premier_2019_20_raw.csv
  - premier_2020_21_raw.csv
  - premier_2021_22_raw.csv
  - premier_2022_23_raw.csv
  - premier_2023_24_raw.csv


### 1.3 Construcción del diccionario de temporadas

Se construye un diccionario que asocia cada temporada con su correspondiente DataFrame.

In [58]:
dfs_all = {}

for f in csv_files:
    season = f.stem.replace('_raw', '').replace(f'{FILE_PREFIX}_', '')  
    dfs_all[season] = pd.read_csv(f)

seasons_sorted = sorted(
    dfs_all.keys(),
    key=lambda s: int(s.split('_')[0])
)

n = len(seasons_sorted)
print(f"Temporadas cargadas: {n}  ({seasons_sorted[0]} → {seasons_sorted[-1]})\n")
print("  |  ".join(seasons_sorted))

Temporadas cargadas: 10  (2014_15 → 2023_24)

2014_15  |  2015_16  |  2016_17  |  2017_18  |  2018_19  |  2019_20  |  2020_21  |  2021_22  |  2022_23  |  2023_24


## 2) Consistencia del esquema de datos 

En esta fase se analiza la coherencia en la estructura del dataset entre temporadas, evaluando la estabilidad de sus dimensiones, variables y tipos de dato.

### 2.1 Dimensiones del dataset por temporada

Se comparan filas y columnas de cada temporada para identificar posibles cambios estructurales.

In [59]:
summary = pd.DataFrame(
    [{
        "Season": s,
        "Rows": dfs_all[s].shape[0],
        "Columns": dfs_all[s].shape[1],
    } for s in seasons_sorted]
)

display(summary.style.hide(axis="index"))

rows_unique = summary["Rows"].nunique()
cols_unique = summary["Columns"].nunique()

if rows_unique == 1:
    print(f"Consistencia en número de filas: todas las temporadas contienen {summary['Rows'].iloc[0]} registros.")
else:
    print("Variación detectada en el número de filas entre temporadas.")

if cols_unique == 1:
    print(f"Consistencia en número de columnas: todas las temporadas contienen {summary['Columns'].iloc[0]} variables.")
else:
    print("Variación detectada en el número de columnas entre temporadas.")

Season,Rows,Columns
2014_15,381,68
2015_16,380,65
2016_17,380,65
2017_18,380,65
2018_19,380,62
2019_20,380,106
2020_21,380,106
2021_22,380,106
2022_23,380,106
2023_24,380,106


Variación detectada en el número de filas entre temporadas.
Variación detectada en el número de columnas entre temporadas.


#### Diagnóstico si hay inconsistencias en filas

In [60]:
if rows_unique > 1:
    print("[!] Diagnóstico de filas inconsistentes:\n")

    for s in seasons_sorted:
        df = dfs_all[s]
        empty_rows = df.isna().all(axis=1).sum()

        if empty_rows > 0:
            print(f"  • {s}: {empty_rows} fila(s) completamente vacía(s) → eliminada(s)")
            dfs_all[s] = df.dropna(how="all")

            for col in dfs_all[s].select_dtypes(include="float64").columns:
                if dfs_all[s][col].notna().all() and (dfs_all[s][col] % 1 == 0).all():
                    dfs_all[s][col] = dfs_all[s][col].astype("int64")

    print(f"\nFilas tras limpieza:")
    for s in seasons_sorted:
        print(f"  • {s}: {len(dfs_all[s])} filas")
else:
    print("Todas las temporadas tienen el mismo número de filas")

[!] Diagnóstico de filas inconsistentes:

  • 2014_15: 1 fila(s) completamente vacía(s) → eliminada(s)

Filas tras limpieza:
  • 2014_15: 380 filas
  • 2015_16: 380 filas
  • 2016_17: 380 filas
  • 2017_18: 380 filas
  • 2018_19: 380 filas
  • 2019_20: 380 filas
  • 2020_21: 380 filas
  • 2021_22: 380 filas
  • 2022_23: 380 filas
  • 2023_24: 380 filas


### 2.2 Intersección de columnas comunes

Se obtiene la intersección de columnas comunes a todas las temporadas.

In [61]:
column_sets = [set(dfs_all[s].columns) for s in seasons_sorted]
common_columns = set.intersection(*column_sets)
print(f"Número de columnas comunes a TODAS las temporadas: {len(common_columns)}")

Número de columnas comunes a TODAS las temporadas: 44


### 2.3 Conjunto de variables comunes (core dataset)

Listado de variables comunes a todo el histórico de temporadas.

#### Resumen global

In [62]:
core_df = pd.DataFrame(sorted(common_columns), columns=["Variable"])
df_class = group_columns(core_df["Variable"])

summary_groups = (
    df_class.groupby("Grupo", as_index=False)
    .agg(**{"Número de variables": ("Variable", "count")})
    .sort_values("Número de variables", ascending=False)
)

display(summary_groups.style.hide(axis="index"))

Grupo,Número de variables
Cuotas de apuestas,21
Estadísticas del partido,12
Resultados y goles,6
Identificación del partido,4
Contexto del partido,1


#### Desagregado por grupos

In [63]:
detail_groups = (
    df_class.sort_values(["Grupo", "Variable"])
    .groupby("Grupo", as_index=False)
    .agg(Variables=("Variable", lambda x: "\n".join(x)))
)

display(
    detail_groups.style
    .set_properties(**{"white-space": "pre-wrap"})
    .hide(axis="index")
)

Grupo,Variables
Contexto del partido,Referee
Cuotas de apuestas,B365A B365D B365H BWA BWD BWH IWA IWD IWH PSA PSCA PSCD PSCH PSD PSH VCA VCD VCH WHA WHD WHH
Estadísticas del partido,AC AF AR AS AST AY HC HF HR HS HST HY
Identificación del partido,AwayTeam Date Div HomeTeam
Resultados y goles,FTAG FTHG FTR HTAG HTHG HTR


### 2.4 Tipos de datos del core dataset

Clasificación de las variables comunes por tipo (numéricas, categóricas y temporales) y validación de sus tipos de dato reales en el dataset.

In [64]:
sample_df = dfs_all[seasons_sorted[0]][sorted(common_columns)]

structure_df = pd.DataFrame({
    "Variable": sample_df.columns,
    "Dtype": sample_df.dtypes.astype(str),
    "Categoría": sample_df.dtypes.apply(
        lambda dt: "Numérica" if pd.api.types.is_numeric_dtype(dt)
        else "Temporal" if pd.api.types.is_datetime64_any_dtype(dt)
        else "Categórica"
    )
})

display(
    structure_df
    .style
    .hide(axis="index")
    .set_table_attributes('style="max-height:400px; overflow-y:auto; display:block;"')
)

print(f"\nPreview de datos — Temporada {seasons_sorted[0]} (5 primeras filas):")
display(sample_df.head(5))

Variable,Dtype,Categoría
AC,int64,Numérica
AF,int64,Numérica
AR,int64,Numérica
AS,int64,Numérica
AST,int64,Numérica
AY,int64,Numérica
AwayTeam,str,Categórica
B365A,float64,Numérica
B365D,float64,Numérica
B365H,float64,Numérica



Preview de datos — Temporada 2014_15 (5 primeras filas):


,AC,AF,AR,AS,AST,AY,AwayTeam,B365A,B365D,B365H,...,PSCH,PSD,PSH,Referee,VCA,VCD,VCH,WHA,WHD,WHH
0,3,19,1,4,2,2,Crystal Palace,15.0,6.5,1.25,...,1.29,6.45,1.26,J Moss,10.50,6.25,1.25,12.0,5.5,1.25
1,6,10,0,13,3,1,Everton,2.4,3.4,3.20,...,3.11,3.38,3.14,M Jones,2.40,3.40,3.20,2.4,3.1,3.10
2,0,20,0,5,4,4,Swansea,11.0,5.0,1.36,...,1.45,5.10,1.37,M Dean,10.00,5.20,1.36,9.0,4.5,1.36
3,9,10,0,11,4,2,Hull,3.1,3.3,2.50,...,2.31,3.26,2.48,C Pawson,3.12,3.20,2.55,2.9,3.0,2.60
4,8,9,0,7,2,3,Aston Villa,4.5,3.5,1.95,...,2.01,3.47,1.95,A Taylor,4.75,3.30,1.95,4.2,3.2,1.95


### 2.5 Cambios en tipos de datos

Detección de columnas cuyo `dtype` (tipo de dato) cambia según la temporada en el core dataset.

In [65]:
drift_candidates = []

for col in common_columns:  
    types_per_season = {s: str(dfs_all[s][col].dtype) for s in seasons_sorted}
    unique_types = set(types_per_season.values())
    
    if len(unique_types) > 1:
        drift_candidates.append({
            "Column": col,
            **types_per_season
        })

drift_df = pd.DataFrame(drift_candidates)

if len(drift_df) > 0:
    print(f"[!] Columnas del core con drift en dtype: {len(drift_df)}")
    display(drift_df.set_index("Column"))
else:
    print("Tipos de datos estables en el core dataset")

Tipos de datos estables en el core dataset


### 2.6 Identificación de columnas numéricas estables

Selección de variables numéricas estables para la posterior validación de valores negativos.

In [66]:
numeric_sets = [
    set(dfs_all[s][sorted(common_columns)].select_dtypes(include="number").columns)
    for s in seasons_sorted
]
core_numeric = sorted(set.intersection(*numeric_sets))
all_numeric_in_core = sorted(set.union(*numeric_sets))
unstable_numeric = sorted(set(all_numeric_in_core) - set(core_numeric))

print(f"Columnas numéricas estables en el core: {len(core_numeric)}")

if len(unstable_numeric) > 0:
    print(f"\n[!] Columnas del core que son numéricas solo en algunas temporadas: {len(unstable_numeric)}")
    print(f"  {', '.join(unstable_numeric)}")
else:
    print("Todas las columnas numéricas del core son estables")

Columnas numéricas estables en el core: 37
Todas las columnas numéricas del core son estables


## 3) Calidad de datos: valores nulos

Se analiza la presencia de valores nulos por variable y temporada en el core dataset.

### 3.1 Construcción del resumen de valores nulos

Se consolida la información de valores nulos en una tabla estructurada por temporada y variable.

In [67]:
null_rows = []

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]
    null_counts = df.isna().sum()
    null_pct = (df.isna().mean() * 100).round(2)
    
    season_nulls = pd.DataFrame({
        "Season": s,
        "Column": df.columns,
        "Nulls": null_counts.values,
        "Null_%": null_pct.values
    })
    null_rows.append(season_nulls)

null_core_df = pd.concat(null_rows, ignore_index=True)
print(f"Nulos detectados en el core dataset: {null_core_df['Nulls'].sum():,}")

Nulos detectados en el core dataset: 558


### 3.2 Columnas con nulos relevantes

Se agregan los resultados para obtener una visión global del nivel de ausencia de datos en cada temporada.

In [68]:
season_summary = (
    null_core_df.groupby("Season", as_index=False)
    .agg(
        **{
            "% nulos medio": ("Null_%", "mean"),
            "Columnas con nulos": ("Nulls", lambda x: (x > 0).sum()),
            "Máx. % nulos": ("Null_%", "max"),
        }
    )
)

season_with_nulls = season_summary[season_summary["Columnas con nulos"] > 0]
display(season_with_nulls.style.hide(axis="index"))

Season,% nulos medio,Columnas con nulos,Máx. % nulos
2014_15,0.017727,3,0.260000
2015_16,0.017727,3,0.260000
2023_24,3.301364,6,47.890000


### 3.3 Filtro de nulos relevantes

Se identifican las variables cuyo porcentaje de nulos supera el umbral definido.

In [69]:
THRESHOLD_NULL_PCT = 5.0

null_relevant = (
    null_core_df[null_core_df["Null_%"] > THRESHOLD_NULL_PCT]
    .sort_values(["Season", "Null_%"], ascending=[True, False])
    .reset_index(drop=True)
)

print(f"Umbral aplicado: > {THRESHOLD_NULL_PCT:.1f}% nulos")
print(f"Filas que superan el umbral: {len(null_relevant)}")

if len(null_relevant) > 0:
    display(null_relevant.style.hide(axis="index"))
else:
    print("Ninguna columna del core supera el umbral")

Umbral aplicado: > 5.0% nulos
Filas que superan el umbral: 3


Season,Column,Nulls,Null_%
2023_24,IWA,182,47.890000
2023_24,IWD,182,47.890000
2023_24,IWH,182,47.890000


### 3.4 Análisis global por variable

Se detectan las variables que presentan mayores niveles de ausencia de datos en el core dataset.

In [70]:
worst_columns = (
    null_core_df.groupby("Column", as_index=False)
    .agg(**{"Máx. % nulos (cualquier temporada)": ("Null_%", "max")})
    .sort_values("Máx. % nulos (cualquier temporada)", ascending=False)
)

worst_columns_filtered = worst_columns[worst_columns["Máx. % nulos (cualquier temporada)"] > 0]

if len(worst_columns_filtered) > 0:
    display(
        worst_columns_filtered
        .style
        .hide(axis="index")
        .set_table_attributes('style="max-height:300px; overflow-y:auto; display:block;"')
    )
else:
    print("Ninguna columna del core presenta nulos en ninguna temporada")

Column,Máx. % nulos (cualquier temporada)
IWA,47.890000
IWD,47.890000
IWH,47.890000
BWD,0.530000
BWH,0.530000
BWA,0.530000


## 4) Validaciones de integridad

Se realizan comprobaciones básicas de coherencia lógica y estructural sobre el core dataset.


### 4.1 Validación de identificador de liga (`Div`)

Verificación de que la columna `Div` contiene exclusivamente el código esperado para esta competición.

In [71]:
expected_div = next(code for code, name in ALL_LEAGUES.items() if name == LEAGUE)

actual_divs = pd.concat([dfs_all[s]["Div"] for s in seasons_sorted]).unique()

if len(actual_divs) == 1 and actual_divs[0] == expected_div:
    print(f"Validación de Div: valor constante '{expected_div}' ({LEAGUE})")
else:
    print(f"[!] Valores inesperados en Div: {actual_divs} (esperado: {expected_div})")

Validación de Div: valor constante 'E0' (premier)


### 4.2 Validación de categorías en resultados

Se comprueba que las variables `FTR` (resultado final) y `HTR` (resultado al descanso) contengan únicamente las categorías esperadas: `H`, `D` y `A`.

In [72]:
expected = {"H", "D", "A"}
issues_found = False

for s in seasons_sorted:
    ftr_vals = set(dfs_all[s]["FTR"].dropna().unique())
    htr_vals = set(dfs_all[s]["HTR"].dropna().unique())
    
    if ftr_vals != expected or htr_vals != expected:
        print(f"[!] {s}: FTR={ftr_vals}, HTR={htr_vals}")
        issues_found = True

if not issues_found:
    print("Validación FTR/HTR: todas las temporadas correctas")

Validación FTR/HTR: todas las temporadas correctas


### 4.3 Consistencia en nombres de equipos

Se valida la consistencia de los nombres de equipos a lo largo de las temporadas del core dataset.

In [73]:
result = check_name_consistency(dfs_all, seasons_sorted, common_columns)

print(f"Equipos únicos: {result['total_teams']}  |  Colisiones: {len(result['collisions'])}\n")

if len(result['collisions']) > 0:
    print("[!] Colisiones detectadas:")
    display(result['collisions'][["Team_norm", "n_variants", "Variants"]].rename(columns={
        "Team_norm": "Nombre de equipos normalizado",
        "n_variants": "Número de variantes",
        "Variants": "Variantes detectadas"
    })
    .style.hide(axis="index")
)
else:
    print("Nombres de equipos consistentes entre temporadas")

teams_by_season = {}
for s in seasons_sorted:
    df_core = dfs_all[s][sorted(common_columns)]
    teams_by_season[s] = set(df_core["HomeTeam"].unique()) | set(df_core["AwayTeam"].unique())

all_teams = sorted({
    str(team).strip()
    for teams in teams_by_season.values()
    for team in teams
    if pd.notna(team) and str(team).strip() != ""
})
teams_df = pd.DataFrame({"Equipo": all_teams})
display(teams_df.style.hide(axis="index").set_table_attributes('style="max-height:300px; overflow-y:auto; display:block;"'))

Equipos únicos: 34  |  Colisiones: 0

Nombres de equipos consistentes entre temporadas


Equipo
Arsenal
Aston Villa
Bournemouth
Brentford
Brighton
Burnley
Cardiff
Chelsea
Crystal Palace
Everton


### 4.4 Detección de duplicados

Se detectan posibles partidos duplicados utilizando `Date`, `HomeTeam` y `AwayTeam` como identificador del mismo.

In [74]:
duplicates_found = False

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]
    dups = df.duplicated(subset=["Date", "HomeTeam", "AwayTeam"], keep=False)
    
    if dups.sum() > 0:
        print(f"[!] {s}: {dups.sum()} filas duplicadas detectadas")
        display(df[dups][["Date", "HomeTeam", "AwayTeam", "FTR"]])
        duplicates_found = True

if not duplicates_found:
    print("Validación de duplicados: ninguna temporada afectada")

Validación de duplicados: ninguna temporada afectada


### 4.5 Coherencia resultado vs goles

Validación de consistencia entre `FTR` y marcadores finales (`FTHG`, `FTAG`).

In [75]:
issues_found = False

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]

    inconsistent = df[
        ((df["FTR"] == "H") & (df["FTHG"] <= df["FTAG"])) |
        ((df["FTR"] == "A") & (df["FTAG"] <= df["FTHG"])) |
        ((df["FTR"] == "D") & (df["FTHG"] != df["FTAG"]))
    ]

    if len(inconsistent) > 0:
        print(f"[!] {s}: {len(inconsistent)} inconsistencias FTR vs goles")
        display(inconsistent[["Date", "HomeTeam", "AwayTeam", "FTHG", "FTAG", "FTR"]])
        issues_found = True

if not issues_found:
    print("Validación de consistencia FTR: todas las temporadas correctas")

Validación de consistencia FTR: todas las temporadas correctas


### 4.6 Control de valores negativos

Control de calidad para columnas numéricas del core dataset donde no se esperan valores negativos.

In [76]:
negative_summary = []
issues_found = False

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]

    for col in core_numeric:
        neg_mask = df[col] < 0
        neg_count = neg_mask.sum()
        
        negative_summary.append({
            "Season": s,
            "Column": col,
            "Negative_values": neg_count
        })
        
        if neg_count > 0:
            print(f"[!] {s} - {col}: {neg_count} valores negativos")
            display(df[neg_mask][["Date", "HomeTeam", "AwayTeam", col]])
            issues_found = True

if not issues_found:
    print("Validación de valores negativos: todas las columnas correctas")

negative_df = pd.DataFrame(negative_summary)

Validación de valores negativos: todas las columnas correctas


## 5) Variables de cuotas

Análisis de columnas asociadas a casas de apuestas para verificar su consistencia, integridad y cobertura en el dataset.

### 5.1 Identificación de columnas de casas de apuestas 

Se identifican las columnas de casas de apuestas presentes en todas las temporadas del dataset.

In [77]:
# Prefijos de casas de apuestas según documentación oficial de football-data
bookmaker_prefixes = [
    "1XB", "B365", "BF", "BFD", "BMGM", "BV", "BS", "BW", 
    "CL", "GB", "IW", "LB", "PS", "SO", "SB", "SJ", 
    "SY", "VC", "WH"
]

odds_columns = [col for col in common_columns 
                if any(col.startswith(p) for p in bookmaker_prefixes)]

bookmakers = {}
for col in odds_columns:
    prefix = next(p for p in bookmaker_prefixes if col.startswith(p))
    bookmakers.setdefault(prefix, []).append(col)

print(f"Casas de apuestas en el core: {len(bookmakers)}  |  Columnas de cuotas: {len(odds_columns)}\n")
for book, cols in sorted(bookmakers.items()):
    print(f"  {book:4s} → {', '.join(sorted(cols))}")

Casas de apuestas en el core: 6  |  Columnas de cuotas: 21

  B365 → B365A, B365D, B365H
  BW   → BWA, BWD, BWH
  IW   → IWA, IWD, IWH
  PS   → PSA, PSCA, PSCD, PSCH, PSD, PSH
  VC   → VCA, VCD, VCH
  WH   → WHA, WHD, WHH


### 5.2 Consistencia de mercados por casa de apuestas

Verificación de que cada casa de apuestas tiene las tres columnas esperadas (H/D/A).

In [78]:
issues_found = False
extra_variants = []

for book in bookmakers:
    cols = set(bookmakers[book])
    
    basic = {f"{book}H", f"{book}D", f"{book}A"}
    if not basic.issubset(cols):
        print(f"[!] {book}: mercado básico H/D/A incompleto")
        issues_found = True
    elif len(cols) > 3:
        extra_variants.append(f"{book} ({', '.join(sorted(cols - basic))})")

if not issues_found:
    print("Todas las casas tienen mercados básicos completos (H/D/A)")
    if extra_variants:
        print(f"\nVariantes adicionales detectadas:")
        for variant in extra_variants:
            print(f"  • {variant}")

Todas las casas tienen mercados básicos completos (H/D/A)

Variantes adicionales detectadas:
  • PS (PSCA, PSCD, PSCH)


### 5.3 Detección de odds inválidas

Identificación de valores fuera de rango esperado (< 1.0 o > 100).

In [79]:
MIN_VALID_ODD = 1.0   # Cuotas < 1.0 son matemáticamente inválidas
MAX_VALID_ODD = 100.0 # Cuotas > 100 son extremadamente raras en ligas principales

issues_found = False

for s in seasons_sorted:
    df = dfs_all[s]
    for col in odds_columns:
        invalid = ((df[col] < MIN_VALID_ODD) | (df[col] > MAX_VALID_ODD)).sum()
        if invalid > 0:
            print(f"[!] {s} - {col}: {invalid} cuotas fuera de rango [{MIN_VALID_ODD}, {MAX_VALID_ODD}]")
            issues_found = True

if not issues_found:
    print(f"Todas las cuotas están en el rango válido [{MIN_VALID_ODD}, {MAX_VALID_ODD}]")

Todas las cuotas están en el rango válido [1.0, 100.0]


### 5.4 Cobertura de odds por partido

Detección de partidos sin ninguna cuota disponible en el dataset.

In [80]:
issues_found = False

for s in seasons_sorted:
    df = dfs_all[s][sorted(common_columns)]
    rows_missing_all_odds = df[odds_columns].isnull().all(axis=1).sum()
    
    if rows_missing_all_odds > 0:
        print(f"[!] {s}: {rows_missing_all_odds} partido(s) sin cuotas")
        
        missing_odds_mask = df[odds_columns].isnull().all(axis=1)
        display(df[missing_odds_mask][["Date", "HomeTeam", "AwayTeam"]])
        
        issues_found = True

if not issues_found:
    print("Todos los partidos tienen al menos una cuota disponible")

Todos los partidos tienen al menos una cuota disponible


## 6) Conclusiones del análisis exploratorio

El dataset estudiado comprende **10 temporadas** (2014/15–2023/24) con un total de **3.800 partidos** de la Premier League. donde se identificó y eliminó 1 fila completamente vacía al final de la temporada 2014/15. La eliminación se realizó durante el EDA para evitar que propagase drift de tipos (int64 → float64) al dataset exportado y a las fases posteriores del pipeline.

Se observó **variabilidad en el número de columnas** entre temporadas, principalmente debido a cambios en la cobertura de variables (especialmente cuotas y campos adicionales). La estructura evoluciona desde **62–68 columnas** (2014/15–2018/19) hasta **106 columnas** (2019/20–2023/24). A partir de esta variabilidad se definió un **core dataset de 44 variables comunes**, presente de forma consistente en todas las temporadas, que constituye la base estable para el análisis longitudinal.

El análisis de los tipos de datos confirmó la **ausencia de drift**: las 44 columnas del core conservan tipos coherentes entre temporadas, con **37 variables numéricas** que permanecen estables.

### Estructura del core dataset

El core dataset se distribuye en:

| Grupo | Variables |
|-------|-----------|
| Identificación (4) | `Date`, `Div`, `HomeTeam`, `AwayTeam` |
| Resultados (6) | `FTHG`, `FTAG`, `FTR`, `HTHG`, `HTAG`, `HTR` |
| Estadísticas (12) | `HS`, `AS`, `HST`, `AST`, `HF`, `AF`, `HC`, `AC`, `HY`, `AY`, `HR`, `AR` |
| Contexto del partido (1) | `Referee` |
| Cuotas (21) | `B365`, `BW`, `IW`, `PS`, `VC`, `WH` (H/D/A + variantes) |


### Calidad y completitud

- Se detectaron **558 valores nulos** en el core dataset.
- Los valores nulos se concentran en **variables de cuotas**, especialmente **Interlive** (`IWA`, `IWD`, `IWH`) 
con un **48% de nulos en 2023/24**, cuya utilidad real se evaluará en la fases posteriores.
- El resto de columnas con nulos presentan máximos muy bajos.
- La estrategia consistirá en **priorizar casas con mayor cobertura** y menor proporción de nulos en fases posteriores.
- Las variables de **resultados y estadísticas de partido** presentan **completitud total**.

### Consideraciones para la fase de limpieza

- Conversión de `Date` a formato `datetime`.
- Transformación de `Div`: identificador de liga con valor constante **E0** en todas las temporadas. Se renombrará con un valor más legible en la fase de limpieza.
- Se verificó **consistencia total en nombres de equipos**: **34 equipos únicos** sin colisiones tras normalización.
- Incorporación de validaciones automáticas: categorías válidas (`FTR`/`HTR`), ausencia de duplicados, rangos esperados en cuotas y estadísticas.

---

En conjunto, el core dataset muestra estabilidad estructural, coherencia interna y alta completitud, constituyendo una base adecuada para la fase de limpieza y posterior modelado.

---

## 7) Exportación del core dataset

### 7.1 Montaje del core dataset 

Concatenación de todas las temporadas con las columnas del core dataset.

In [81]:
core_columns = core_df["Variable"].tolist()

for s in seasons_sorted:
    missing = set(core_columns) - set(dfs_all[s].columns)
    if missing:
        raise ValueError(f"[!] {s}: columnas faltantes en core: {missing}")

df_core_all = pd.concat(
    [dfs_all[s][core_columns] for s in seasons_sorted],
    ignore_index=True
)

print(f"Core dataset montado:")
print(f"  Filas: {len(df_core_all):,} | Columnas: {len(core_columns)}")

Core dataset montado:
  Filas: 3,800 | Columnas: 44


### 7.2 Exportación a Parquet y metadatos

Guardado del core dataset en formato Parquet con archivo JSON de metadatos del esquema.

In [82]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df_core_all.to_parquet(CORE_DIR, index=False)

metadata = {
    "num_columns": len(core_columns),
    "num_rows": len(df_core_all),
    "num_seasons": len(seasons_sorted),
    "seasons": seasons_sorted,
    "columns": core_columns,
    "dtypes": {col: str(df_core_all[col].dtype) for col in core_columns}
}

with open(METADATA_DIR, 'w') as f:
    json.dump(metadata, f, indent=2)
    
core_rel = CORE_DIR.relative_to(PROJECT_ROOT)
metadata_rel = METADATA_DIR.relative_to(PROJECT_ROOT)

print(f"Archivos guardados:")
print(f"  · Dataset -> {core_rel}")
print(f"  · Metadatos -> {metadata_rel}")

Archivos guardados:
  · Dataset -> data/processed/premier/core_validated.parquet
  · Metadatos -> data/processed/premier/core_schema.json
